# Visualize Building Thresholds in t-SNE Latent Space

This notebook visualizes the classified building thresholds (b1-b4) alongside the original 8 typologies in the MAE latent space using t-SNE.

**Pipeline:**
- Load fine-tuned MAE encoder
- Extract embeddings for original typologies (t1-t8)
- Extract embeddings for building thresholds (b1-b4)
- Compute t-SNE on combined embeddings
- Visualize with typologies as colored circles, buildings as distinct markers
- Export as SVG

## Setup

In [ ]:
import sys
sys.path.append('../models_MAE')

import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.manifold import TSNE

from config_mae import get_mae_config
from timesformer_mae import TimeSformerMAE
from dataset_mae import ThresholdSequenceDatasetMAE
from dataset_classifier import BuildingDataset

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f"Device: {device}")

## Configuration

In [ ]:
# Paths - Update these to match your local setup
DATA_ROOT = '../data'

# Try full checkpoint first, fall back to encoder-only weights
CHECKPOINT_PATH = '../models_MAE/output_mae_adjusted/checkpoints/best_model.pt'
ENCODER_PATH = '../models_MAE/output_mae_adjusted/encoder_mae_pretrained.pt'

OUTPUT_DIR = '../output_classifier/tsne_buildings'

# Building data
PANOS_BUILDINGS_DIR = os.path.join(DATA_ROOT, 'panos_buildings')
CANDIDATES_DIR = os.path.join(DATA_ROOT, 'candidates')
BUILDING_IDS = ['b1', 'b2', 'b3', 'b4']

# Typology names
TYPOLOGY_NAMES = ['t1', 't2', 't3', 't4', 't5', 't6', 't7', 't8']

# t-SNE parameters
PERPLEXITY = 30
RANDOM_STATE = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Configuration:")
print(f"  Data root: {DATA_ROOT}")
print(f"  Checkpoint: {CHECKPOINT_PATH}")
print(f"  Encoder: {ENCODER_PATH}")
print(f"  Buildings: {BUILDING_IDS}")
print(f"  Output: {OUTPUT_DIR}")

## Load MAE Encoder

In [ ]:
print("Loading MAE encoder...")

# Load config and model
config = get_mae_config()
model = TimeSformerMAE(config)

# Try to load full checkpoint first, fall back to encoder-only weights
if os.path.exists(CHECKPOINT_PATH):
    print(f"Loading full checkpoint from: {CHECKPOINT_PATH}")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"  Trained epoch: {checkpoint['epoch']+1}")
    print(f"  Val loss: {checkpoint['val_loss']:.4f}")
elif os.path.exists(ENCODER_PATH):
    print(f"Loading encoder weights from: {ENCODER_PATH}")
    encoder_weights = torch.load(ENCODER_PATH, map_location=device, weights_only=False)
    # Load partial state dict (encoder weights only)
    model.load_state_dict(encoder_weights, strict=False)
    print(f"  Loaded encoder-only weights (decoder weights are random)")
else:
    raise FileNotFoundError(f"Neither {CHECKPOINT_PATH} nor {ENCODER_PATH} found!")

model = model.to(device)
model.eval()

print(f"✓ Model loaded successfully!")
print()

## Load Original Typology Dataset

In [ ]:
print("Loading original typology dataset...")

# Find all typology CSV files (exclude building CSVs)
candidates_dir = os.path.join(DATA_ROOT, 'candidates')
csv_files = [os.path.join(candidates_dir, f) for f in os.listdir(candidates_dir)
             if f.endswith('.csv') and not f.startswith('b')]
csv_files = sorted(csv_files)

pano_folder = os.path.join(DATA_ROOT, 'panos')

# Create dataset
typology_dataset = ThresholdSequenceDatasetMAE(
    metadata_csvs=csv_files,
    pano_folder=pano_folder
)

print(f"✓ Loaded {len(typology_dataset)} typology threshold sequences")
print()

## Load Building Dataset

In [ ]:
print("Loading building thresholds...")

building_dataset = BuildingDataset(
    panos_dir=PANOS_BUILDINGS_DIR,
    candidates_dir=CANDIDATES_DIR,
    building_ids=BUILDING_IDS
)

print(f"\n✓ Loaded {len(building_dataset)} building threshold sequences")
print()

## Extract Embeddings for Original Typologies

In [ ]:
print("="*80)
print("EXTRACTING EMBEDDINGS FOR ORIGINAL TYPOLOGIES")
print("="*80)
print()

# Sample from each typology (to keep visualization manageable)
SAMPLES_PER_TYPOLOGY = 150  # Adjust as needed

# Group samples by typology
typology_indices = {i: [] for i in range(8)}
for idx in range(len(typology_dataset)):
    sample = typology_dataset[idx]
    label = sample['typology_label']
    typology_indices[label].append(idx)

# Sample indices
np.random.seed(RANDOM_STATE)
selected_indices = []
for label in range(8):
    indices = typology_indices[label]
    n_samples = min(SAMPLES_PER_TYPOLOGY, len(indices))
    selected = np.random.choice(indices, n_samples, replace=False)
    selected_indices.extend(selected)
    print(f"  {TYPOLOGY_NAMES[label]}: {n_samples} samples")

print(f"\nTotal typology samples: {len(selected_indices)}")
print()

# Extract embeddings
typology_embeddings = []
typology_labels = []
typology_names_list = []

print("Extracting embeddings...")
with torch.no_grad():
    for idx in tqdm(selected_indices):
        sample = typology_dataset[idx]
        frames = sample['frames'].unsqueeze(0).to(device)  # (1, 7, 1, 32, 64)
        
        # Run encoder WITHOUT masking
        x_encoded, _, _ = model.forward_encoder(frames, frame_mask_ratios=(0,0,0,0,0,0,0))
        
        # Global average pooling
        h = x_encoded.mean(dim=1)  # (1, 384)
        
        typology_embeddings.append(h.cpu().numpy())
        typology_labels.append(sample['typology_label'])
        typology_names_list.append(sample['metadata']['typology'].split('-')[0])

typology_embeddings = np.concatenate(typology_embeddings, axis=0)
typology_labels = np.array(typology_labels)

print(f"\n✓ Extracted {len(typology_embeddings)} typology embeddings")
print(f"  Shape: {typology_embeddings.shape}")
print()

## Extract Embeddings for Buildings

In [ ]:
print("="*80)
print("EXTRACTING EMBEDDINGS FOR BUILDINGS")
print("="*80)
print()

# Load predictions to get classified typologies
predictions_path = '../output_classifier/building_predictions/all_predictions.csv'
predictions_df = pd.read_csv(predictions_path)

print(f"Loaded {len(predictions_df)} building predictions")
print()

# Extract embeddings for all building thresholds
building_embeddings = []
building_ids_list = []
building_threshold_ids = []
building_predicted_classes = []

print("Extracting embeddings...")
with torch.no_grad():
    for idx in tqdm(range(len(building_dataset))):
        sample = building_dataset[idx]
        frames = sample['frames'].unsqueeze(0).to(device)  # (1, 7, 1, 32, 64)
        
        # Run encoder WITHOUT masking
        x_encoded, _, _ = model.forward_encoder(frames, frame_mask_ratios=(0,0,0,0,0,0,0))
        
        # Global average pooling
        h = x_encoded.mean(dim=1)  # (1, 384)
        
        building_embeddings.append(h.cpu().numpy())
        building_ids_list.append(sample['building_id'])
        building_threshold_ids.append(sample['threshold_id'])
        
        # Get predicted class from predictions CSV
        pred_row = predictions_df[predictions_df['threshold_id'] == sample['threshold_id']]
        if len(pred_row) > 0:
            building_predicted_classes.append(pred_row.iloc[0]['predicted_class'])
        else:
            building_predicted_classes.append('unknown')

building_embeddings = np.concatenate(building_embeddings, axis=0)

print(f"\n✓ Extracted {len(building_embeddings)} building embeddings")
print(f"  Shape: {building_embeddings.shape}")
print()

# Summary per building
for building_id in BUILDING_IDS:
    count = sum(1 for bid in building_ids_list if bid == building_id)
    print(f"  {building_id}: {count} thresholds")
print()

## Compute t-SNE on Combined Embeddings

In [ ]:
print("="*80)
print("COMPUTING t-SNE")
print("="*80)
print()

# Combine all embeddings
all_embeddings = np.vstack([typology_embeddings, building_embeddings])
print(f"Combined embeddings shape: {all_embeddings.shape}")
print()

# Compute t-SNE
print("Computing t-SNE (this may take a few minutes)...")
tsne = TSNE(
    n_components=2, 
    random_state=RANDOM_STATE, 
    perplexity=min(PERPLEXITY, len(all_embeddings) // 4),
    n_iter=1000,
    init='pca'
)
all_embeddings_2d = tsne.fit_transform(all_embeddings)

# Split back
n_typology = len(typology_embeddings)
typology_2d = all_embeddings_2d[:n_typology]
building_2d = all_embeddings_2d[n_typology:]

print(f"\n✓ t-SNE complete!")
print(f"  Typology points: {len(typology_2d)}")
print(f"  Building points: {len(building_2d)}")
print()

## Visualize t-SNE: Typologies + Buildings

In [ ]:
print("="*80)
print("CREATING VISUALIZATION")
print("="*80)
print()

# Color palette for typologies
typology_colors = plt.cm.tab10(np.linspace(0, 1, 10))[:8]

# Markers for buildings
building_markers = {
    'b1': 's',  # square
    'b2': '^',  # triangle up
    'b3': 'D',  # diamond
    'b4': 'v'   # triangle down
}

building_colors = {
    'b1': '#FF0000',  # red
    'b2': '#00FF00',  # green
    'b3': '#0000FF',  # blue
    'b4': '#FF00FF'   # magenta
}

fig, ax = plt.subplots(1, 1, figsize=(16, 12))

# Plot original typologies as background
for i, typ_name in enumerate(TYPOLOGY_NAMES):
    mask = typology_labels == i
    ax.scatter(
        typology_2d[mask, 0],
        typology_2d[mask, 1],
        c=[typology_colors[i]],
        label=f'{typ_name} (original)',
        alpha=0.3,
        s=30,
        edgecolors='none',
        marker='o'
    )

# Plot building thresholds with distinct markers
for building_id in BUILDING_IDS:
    mask = [bid == building_id for bid in building_ids_list]
    if sum(mask) > 0:
        building_points = building_2d[mask]
        ax.scatter(
            building_points[:, 0],
            building_points[:, 1],
            c=building_colors[building_id],
            label=f'{building_id.upper()} ({sum(mask)} thresholds)',
            alpha=0.8,
            s=80,
            edgecolors='black',
            linewidths=1,
            marker=building_markers[building_id]
        )

ax.set_xlabel('t-SNE Dimension 1', fontsize=14, fontweight='bold')
ax.set_ylabel('t-SNE Dimension 2', fontsize=14, fontweight='bold')
ax.set_title(
    'Building Thresholds in MAE Latent Space\n(with Original 8 Typologies)',
    fontsize=16,
    fontweight='bold',
    pad=20
)
ax.legend(loc='best', fontsize=11, framealpha=0.95, edgecolor='black')
ax.grid(True, alpha=0.3)
ax.set_facecolor('#f8f9fa')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'tsne_buildings_typologies.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Saved to: {os.path.join(OUTPUT_DIR, 'tsne_buildings_typologies.png')}")
print()

## Color Buildings by Predicted Typology

In [ ]:
# Alternative visualization: color buildings by their predicted typology
fig, ax = plt.subplots(1, 1, figsize=(16, 12))

# Plot original typologies
for i, typ_name in enumerate(TYPOLOGY_NAMES):
    mask = typology_labels == i
    ax.scatter(
        typology_2d[mask, 0],
        typology_2d[mask, 1],
        c=[typology_colors[i]],
        label=f'{typ_name}',
        alpha=0.3,
        s=30,
        edgecolors='none',
        marker='o'
    )

# Plot buildings colored by predicted typology
for building_id in BUILDING_IDS:
    for pred_class in TYPOLOGY_NAMES:
        mask = [(bid == building_id and pred == pred_class) 
                for bid, pred in zip(building_ids_list, building_predicted_classes)]
        if sum(mask) > 0:
            building_points = building_2d[mask]
            typ_idx = TYPOLOGY_NAMES.index(pred_class)
            ax.scatter(
                building_points[:, 0],
                building_points[:, 1],
                c=[typology_colors[typ_idx]],
                alpha=0.9,
                s=100,
                edgecolors='black',
                linewidths=1.5,
                marker=building_markers[building_id]
            )

# Add building markers to legend
for building_id in BUILDING_IDS:
    ax.scatter([], [], c='gray', marker=building_markers[building_id], 
               s=100, label=f'{building_id.upper()}', edgecolors='black', linewidths=1.5)

ax.set_xlabel('t-SNE Dimension 1', fontsize=14, fontweight='bold')
ax.set_ylabel('t-SNE Dimension 2', fontsize=14, fontweight='bold')
ax.set_title(
    'Building Thresholds Colored by Predicted Typology\n(Marker shape = Building, Color = Predicted Typology)',
    fontsize=16,
    fontweight='bold',
    pad=20
)
ax.legend(loc='best', fontsize=11, framealpha=0.95, edgecolor='black', ncol=2)
ax.grid(True, alpha=0.3)
ax.set_facecolor('#f8f9fa')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'tsne_buildings_by_prediction.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Saved to: {os.path.join(OUTPUT_DIR, 'tsne_buildings_by_prediction.png')}")
print()

## Export as SVG

In [ ]:
print("="*80)
print("EXPORTING SVG")
print("="*80)
print()

# Normalize coordinates for SVG
x_min, x_max = all_embeddings_2d[:, 0].min(), all_embeddings_2d[:, 0].max()
y_min, y_max = all_embeddings_2d[:, 1].min(), all_embeddings_2d[:, 1].max()

# SVG dimensions
svg_width = 1000
svg_height = 800
padding = 50

def normalize_coord(x, y):
    nx = padding + (x - x_min) / (x_max - x_min) * (svg_width - 2*padding)
    ny = padding + (y - y_min) / (y_max - y_min) * (svg_height - 2*padding)
    return nx, ny

# Color mapping for typologies (hex)
typology_hex_colors = {
    't1': '#1f77b4',
    't2': '#ff7f0e',
    't3': '#2ca02c',
    't4': '#d62728',
    't5': '#9467bd',
    't6': '#8c564b',
    't7': '#e377c2',
    't8': '#7f7f7f'
}

# Create SVG content
svg_content = f'''<?xml version="1.0" encoding="UTF-8"?>
<svg xmlns="http://www.w3.org/2000/svg" width="{svg_width}" height="{svg_height}" viewBox="0 0 {svg_width} {svg_height}">
  <rect width="100%" height="100%" fill="#f8f9fa"/>
  
  <!-- Original Typologies (circles) -->
  <g id="typologies">
'''

# Add typology points
for i in range(len(typology_2d)):
    x, y = normalize_coord(typology_2d[i, 0], typology_2d[i, 1])
    typ_name = typology_names_list[i]
    color = typology_hex_colors[typ_name]
    svg_content += f'    <circle cx="{x:.2f}" cy="{y:.2f}" r="3" fill="{color}" opacity="0.3" data-typology="{typ_name}"/>\n'

svg_content += '  </g>\n\n  <!-- Building Thresholds -->\n'

# Add building points with different shapes
for building_id in BUILDING_IDS:
    svg_content += f'  <g id="{building_id}">\n'
    
    for i in range(len(building_2d)):
        if building_ids_list[i] == building_id:
            x, y = normalize_coord(building_2d[i, 0], building_2d[i, 1])
            pred_class = building_predicted_classes[i]
            color = typology_hex_colors.get(pred_class, '#000000')
            threshold_id = building_threshold_ids[i]
            
            # Different shapes for different buildings
            if building_id == 'b1':  # square
                svg_content += f'    <rect x="{x-4:.2f}" y="{y-4:.2f}" width="8" height="8" fill="{color}" stroke="black" stroke-width="1" opacity="0.9" data-building="{building_id}" data-threshold="{threshold_id}" data-predicted="{pred_class}"/>\n'
            elif building_id == 'b2':  # triangle up
                svg_content += f'    <polygon points="{x:.2f},{y-5:.2f} {x-5:.2f},{y+4:.2f} {x+5:.2f},{y+4:.2f}" fill="{color}" stroke="black" stroke-width="1" opacity="0.9" data-building="{building_id}" data-threshold="{threshold_id}" data-predicted="{pred_class}"/>\n'
            elif building_id == 'b3':  # diamond
                svg_content += f'    <polygon points="{x:.2f},{y-5:.2f} {x+5:.2f},{y:.2f} {x:.2f},{y+5:.2f} {x-5:.2f},{y:.2f}" fill="{color}" stroke="black" stroke-width="1" opacity="0.9" data-building="{building_id}" data-threshold="{threshold_id}" data-predicted="{pred_class}"/>\n'
            elif building_id == 'b4':  # triangle down
                svg_content += f'    <polygon points="{x:.2f},{y+5:.2f} {x-5:.2f},{y-4:.2f} {x+5:.2f},{y-4:.2f}" fill="{color}" stroke="black" stroke-width="1" opacity="0.9" data-building="{building_id}" data-threshold="{threshold_id}" data-predicted="{pred_class}"/>\n'
    
    svg_content += '  </g>\n'

svg_content += '</svg>'

# Save SVG
svg_path = os.path.join(OUTPUT_DIR, 'tsne_buildings_typologies.svg')
with open(svg_path, 'w') as f:
    f.write(svg_content)

print(f"✓ Saved SVG to: {svg_path}")
print()

# Also export coordinates as CSV for reference
export_data = []

# Typology data
for i in range(len(typology_2d)):
    export_data.append({
        'id': f'typ_{i}',
        'type': 'typology',
        'x': typology_2d[i, 0],
        'y': typology_2d[i, 1],
        'typology': typology_names_list[i],
        'building_id': '',
        'threshold_id': '',
        'predicted_class': typology_names_list[i]  # ground truth for typologies
    })

# Building data
for i in range(len(building_2d)):
    export_data.append({
        'id': f'bld_{i}',
        'type': 'building',
        'x': building_2d[i, 0],
        'y': building_2d[i, 1],
        'typology': building_predicted_classes[i],
        'building_id': building_ids_list[i],
        'threshold_id': building_threshold_ids[i],
        'predicted_class': building_predicted_classes[i]
    })

export_df = pd.DataFrame(export_data)
csv_path = os.path.join(OUTPUT_DIR, 'tsne_coordinates_all.csv')
export_df.to_csv(csv_path, index=False)

print(f"✓ Saved coordinates to: {csv_path}")
print(f"  Total points: {len(export_df)}")
print(f"    - Typology: {len(typology_2d)}")
print(f"    - Building: {len(building_2d)}")
print()

## Summary Statistics

In [ ]:
print("="*80)
print("SUMMARY")
print("="*80)
print()

print("Building Classification Summary:")
for building_id in BUILDING_IDS:
    building_preds = [pred for bid, pred in zip(building_ids_list, building_predicted_classes) if bid == building_id]
    if building_preds:
        pred_counts = pd.Series(building_preds).value_counts()
        top_pred = pred_counts.index[0]
        top_count = pred_counts.iloc[0]
        pct = top_count / len(building_preds) * 100
        print(f"  {building_id.upper()}: Top prediction = {top_pred} ({top_count}/{len(building_preds)} = {pct:.1f}%)")
print()

print("Output Files:")
print(f"  PNG (buildings by ID): {os.path.join(OUTPUT_DIR, 'tsne_buildings_typologies.png')}")
print(f"  PNG (buildings by prediction): {os.path.join(OUTPUT_DIR, 'tsne_buildings_by_prediction.png')}")
print(f"  SVG: {os.path.join(OUTPUT_DIR, 'tsne_buildings_typologies.svg')}")
print(f"  CSV: {os.path.join(OUTPUT_DIR, 'tsne_coordinates_all.csv')}")
print()
print("="*80)
print("\n✓ Visualization complete!")